In [ ]:
import pandas as pd
import re
import os

# File paths
input_path = "/News.csv"
output_path = "/CleanedNews.csv"
df = pd.read_csv(input_path)

def clean(text):
    text = str(text).lower()

    # Remove: LOCATION (Reuters)
    text = re.sub(r'^.*?\(reuters\)\s*[-—:]', '', text)
    text = re.sub(r'^\s*\w+\s+[-—:]', '', text)

    # Remove @ handles and website links
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'http\S+', '', text)

    filteredWords = [
        'said', 'told', 'reported', 'spokesman', 'according',
        'image', 'images', 'via', 'featured', 'video', 'watch', 'read',
        'reuters', 'facebook', 'twitter', 'getty', 'photo', 'by',
        'screenshot', 'screen capture', 'screencapture'
    ]
    pattern = re.compile(r'\b(' + '|'.join(filteredWords) + r')\b')
    text = pattern.sub('', text)

    # Remove single letters, punctuation and whitespace
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['text'] = df['text'].apply(clean)
df.to_csv(output_path, index=False)

print(f"CSV saved to: {output_path}")

CSV saved to: /CleanedNews.csv


In [ ]:
import os
import pandas as pd
import re
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import joblib

# File path
dataset = "/CleanedNews.csv"

# Load CSV
data = pd.read_csv(dataset,index_col=0)
data.head()
data = data.drop(["title", "subject","date"], axis = 1)

# Shuffle data
data = data.sample(frac=1)
data.reset_index(inplace=True)

def preprocess_text(text_data):
    stop_words = stopwords.words('english')
    preprocessed_text = []
    for sentence in text_data:
        sentence = re.sub(r'[^\w\s]', '', str(sentence))
        preprocessed_text.append(' '.join(token.lower()
                                  for token in str(sentence).split()
                                  if token not in stop_words))
    return preprocessed_text

preprocessed_review = preprocess_text(data['text'].values)
data['text'] = preprocessed_review

def get_top_n_words(corpus, n=None):
    vec = CountVectorizer().fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx])
                  for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1],
                        reverse=True)
    return words_freq[:n]

x_train, x_test, y_train, y_test = train_test_split(data['text'],
                                                    data['class'],
                                                    test_size=0.20)

vectorization = TfidfVectorizer()
x_train = vectorization.fit_transform(x_train)
x_test = vectorization.transform(x_test)

model = DecisionTreeClassifier(max_depth=15, min_samples_leaf=220, criterion='entropy')
model.fit(x_train, y_train)

joblib.dump(model, "/fake_news_model.joblib")
joblib.dump(vectorization, "/tfidf_vectorizer.joblib")

# testing the model
print(accuracy_score(y_train, model.predict(x_train)))
print(accuracy_score(y_test, model.predict(x_test)))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0.8647279810769445
0.8590828138913624


In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# 1. Load the pre-cleaned dataset
# We use the 'text' column for features and 'class' for the target
dataset_path = "/CleanedNews.csv"
data = pd.read_csv(dataset_path)

# Ensure no empty rows survived the cleaning process
data = data.dropna(subset=['text', 'class'])

# 2. Split into Training and Testing sets
# We tune on the training set and keep the test set "blind" until the very end
X_train, X_test, y_train, y_test = train_test_split(
    data['text'],
    data['class'],
    test_size=0.20,
    random_state=42
)

# 3. Create a Pipeline
# This treats the Vectorizer and the Classifier as a single object
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('dtree', DecisionTreeClassifier(random_state=42))
])

# 4. Define the Hyperparameter Grid
# Format: 'stepName__parameterName'
param_grid = {
    'dtree__max_depth': [10,15],      # Controls tree complexity
    'dtree__min_samples_leaf': [220,440,880],  # Prevents overfitting to rare words
    'dtree__criterion': ['gini', 'entropy']      # Math used for splitting nodes
}

# 5. Initialize GridSearchCV
# cv=5: Performs 5-fold cross-validation
# n_jobs=-1: Uses all available CPU cores to speed up the search
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# 6. Execute the Search
print("Starting Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

# 7. Results and Evaluation
print("\n--- Tuning Results ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

# Evaluate the best version on the 20% held-out test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- Final Test Set Performance ---")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Starting Hyperparameter Tuning...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

--- Tuning Results ---
Best Parameters: {'dtree__criterion': 'entropy', 'dtree__max_depth': 15, 'dtree__min_samples_leaf': 220}
Best CV Accuracy: 0.8570

--- Final Test Set Performance ---
Test Accuracy: 0.8631
              precision    recall  f1-score   support

           0       0.87      0.86      0.87      4565
           1       0.85      0.87      0.86      4275

    accuracy                           0.86      8840
   macro avg       0.86      0.86      0.86      8840
weighted avg       0.86      0.86      0.86      8840

